# load packages

In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import re

In [ ]:
from datetime import timedelta

In [ ]:
import os

In [ ]:
import seaborn as sns

In [ ]:
import matplotlib.pyplot as plt

# read in files and create df

## simulations

In [ ]:
dfs = []

for i in list(range(1, 81)):
    
    if i in [1, 9, 17, 25, 33, 41, 49, 57, 65, 73]:
        pval = 'none'
    elif i in [2, 10, 18, 26, 34, 42, 50, 58, 66, 74]:
        pval = '0.05'
    elif i in [3, 11, 19, 27, 35, 43, 51, 59, 67, 75]:
        pval = '0.01'
    elif i in [4, 12, 20, 28, 36, 44, 52, 60, 68, 76]:
        pval = '0.001'
    elif i in [5, 13, 21, 29, 37, 45, 53, 61, 69, 77]:
        pval = '0.0001'
    elif i in [6, 14, 22, 30, 38, 46, 54, 62, 70, 78]:
        pval = '0.00001'
    elif i in [7, 15, 23, 31, 39, 47, 55, 63, 71, 79]:
        pval = '0.000001'
    elif i in [8, 16, 24, 32, 40, 48, 56, 64, 72, 80]:
        pval = '0.0000001'

    if i in list(range(1, 17)):
        sim = 'negative_control'
        features_path = ''
    elif i in list(range(17, 33)):
        sim = 'positive_control'
        features_path = 'features_0-1_signal.'
        features = {"feature_0", "feature_1"}
    elif i in list(range(33, 49)):
        sim = 'positive_control'
        features_path = 'features_0-2_signal.'
        features = {"feature_0", "feature_1", "feature_2"}
    elif i in list(range(49, 65)):
        sim = 'positive_control'
        features_path = 'features_0-3_signal.'
        features = {"feature_0", "feature_1", "feature_2", "feature_3"}
    elif i in list(range(65, 81)):
        sim = 'positive_control'
        features_path = 'features_0-4_signal.'
        features = {"feature_0", "feature_1", "feature_2", "feature_3", "feature_4"}

    if i in list(range(1, 9)) + list(range(17, 25)) + list(range(33, 41)) + list(range(49, 57)) + list(range(65, 73)):
        grammar = 'genn'
    else:
        grammar = 'gesr'

    input_dir = 'ML/athena/output/simulations/'
    input_file = 'ADSP.simulated.pval_' + pval + '.' + features_path + 'keep_quest_comb.' + sim + '.' + grammar + '_summary.txt'
    
    try:
        fitness_df = pd.read_csv(input_dir + input_file,
                                 sep = '\t',
                                 nrows = 5)
        rep_df = pd.read_csv(input_dir + input_file,
                             sep = '\t',
                             skiprows = 7,
                             nrows = 5)
        alt_train_df = pd.read_csv(input_dir + input_file,
                                   sep = '\t',
                                   skiprows = 15,
                                   nrows = 5)
        alt_test_df = pd.read_csv(input_dir + input_file,
                                  sep = '\t',
                                  skiprows = 24,
                                  nrows = 5)

        train_ba = fitness_df['balanced_acc Training'].mean()
        test_ba = fitness_df['Testing'].mean()

        train_auroc = alt_train_df['auc'].mean()
        test_auroc = alt_test_df['auc'].mean()

        train_auprc = alt_train_df['auprc'].mean()
        test_auprc = alt_test_df['auprc'].mean()

        train_f1 = alt_train_df['f1_score'].mean()
        test_f1 = alt_test_df['f1_score'].mean()

        important = rep_df[['CV count', 'Variables']]
        important['Variables'] = important['Variables'].str.split(' ')
        important = important.explode('Variables')
        important = important[important['Variables'].str.strip() != '']

        important_dict = important['CV count'].value_counts(dropna = False).to_dict()
        for cv in list(range(1, 6)):
            if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
                important_dict[cv] = np.nan
            else:
                continue
        important_dict = dict(sorted(important_dict.items()))
        important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

        if sim == 'negative_control':
            all_bool = np.nan
            important_bool = np.nan
            nonexpected_important_bool = np.nan
            expected_dict = np.nan
        else:
            all_bool = features.issubset(important["Variables"])
            important_90 = important[important['CV count'] >= 4]
            important_bool = features.issubset(important_90["Variables"])
            nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)
            expected = important[important['Variables'].isin(features)]
            expected_dict = dict(zip(expected['Variables'], expected['CV count']))
            expected_dict = dict(sorted(expected_dict.items()))

        df = pd.DataFrame({'Simulated Data' : [features_path + sim],
                           'pval' : [pval],
                           'grammar' : [grammar],
                           'training BA' : [train_ba],
                           'testing BA' : [test_ba],
                           'training AUROC' : [train_auroc],
                           'testing AUROC' : [test_auroc],
                           'training AUPRC' : [train_auprc],
                           'testing AUPRC' : [test_auprc],
                           'training F1' : [train_f1],
                           'testing F1' : [test_f1],
                           'N feature replications' : [important_dict],
                           'N expected feature replications': [expected_dict],
                           'N features in 80% CVs' : [important_90_n],
                           'expected important features identified in any amount of CVs' : [all_bool],
                           'expected important features identified in 80% CVs' : [important_bool],
                           'non-expected important features identified in 80% CVs' : [nonexpected_important_bool]})
    except FileNotFoundError:
        print("Skipping missing file: " + str(i))

        df = pd.DataFrame({'Simulated Data' : [features_path + sim],
                           'pval' : [pval],
                           'grammar' : [grammar],
                           'training BA' : [np.nan],
                           'testing BA' : [np.nan],
                           'training AUROC' : [np.nan],
                           'testing AUROC' : [np.nan],
                           'training AUPRC' : [np.nan],
                           'testing AUPRC' : [np.nan],
                           'training F1' : [np.nan],
                           'testing F1' : [np.nan],
                           'N feature replications' : [np.nan],
                           'N expected feature replications': [np.nan],
                           'N features in 80% CVs' : [np.nan],
                           'expected important features identified in any amount of CVs' : [np.nan],
                           'expected important features identified in 80% CVs' : [np.nan],
                           'non-expected important features identified in 80% CVs' : [np.nan]})
    dfs.append(df)

sim_df_cat = pd.concat(dfs, axis = 0)
print(sim_df_cat.shape)
sim_df_cat

# read in other input files

## simulated eval metrics

In [ ]:
dfs = []
for pval in ['none', '0.05', '0.01', '0.001', '0.0001', '0.00001', '0.000001', '0.0000001']:
    
    input_dir = 'ML/simulated_datasets/'
    input_file = 'ADSP.simulated.keep_quest_comb.pval_' + pval + '.eval_metrics.txt'
    
    df = pd.read_csv(input_dir + input_file,
                     sep = '\t')

    df.insert(1, 'pval', pval)
    df['Simulated Data'] = df['Simulated Data'].str.replace('6791 Features, ', '')
    df['Simulated Data'] = df['Simulated Data'].str.replace('- 6791 Features', '')

    dfs.append(df)

eval_df = pd.concat(dfs, axis = 0)
print(eval_df.shape)
eval_df

# convert to wide form

## simulated

In [ ]:
order_df = sim_df_cat[['Simulated Data', 'pval']].drop_duplicates()
order_df['order'] = range(len(order_df))

In [ ]:
sim_wide = sim_df_cat.pivot(index = ['Simulated Data', 'pval'], columns = 'grammar')
sim_wide.columns = ['_'.join(col).strip() for col in sim_wide.columns]
sim_wide = sim_wide.reset_index()
sim_wide = sim_wide.merge(order_df, on = ['Simulated Data', 'pval'], how = 'inner')
sim_wide = sim_wide.sort_values('order').drop(columns = 'order')
print(sim_wide.shape)
sim_wide

# extract variables

## avg pathway

### standard scaled

In [ ]:
fitness_df = avg_pathway_standard_genn_fitness.copy()
rep_df = avg_pathway_standard_genn_rep.copy()
alt_train_df = avg_pathway_standard_genn_alt_train.copy()
alt_test_df = avg_pathway_standard_genn_alt_test.copy()

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             important_90_n]
important[important['CV count'] >= 3]

In [ ]:
fitness_df = avg_pathway_standard_gesr_fitness.copy()
rep_df = avg_pathway_standard_gesr_rep.copy()
alt_train_df = avg_pathway_standard_gesr_alt_train.copy()
alt_test_df = avg_pathway_standard_gesr_alt_test.copy()

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             important_90_n]
important[important['CV count'] >= 3]

In [ ]:
avg_pathway_standard_list = ['Avg Pathway- Standard Scaled'] + genn_list + gesr_list

### minmax scaled

In [ ]:
fitness_df = avg_pathway_minmax_genn_fitness.copy()
rep_df = avg_pathway_minmax_genn_rep.copy()
alt_train_df = avg_pathway_minmax_genn_alt_train.copy()
alt_test_df = avg_pathway_minmax_genn_alt_test.copy()

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])


genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             important_90_n]
important[important['CV count'] >= 3]

In [ ]:
fitness_df = avg_pathway_minmax_gesr_fitness.copy()
rep_df = avg_pathway_minmax_gesr_rep.copy()
alt_train_df = avg_pathway_minmax_gesr_alt_train.copy()
alt_test_df = avg_pathway_minmax_gesr_alt_test.copy()

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             important_90_n]
important[important['CV count'] >= 3]

In [ ]:
avg_pathway_minmax_list = ['Avg Pathway- Minmax Scaled'] + genn_list + gesr_list

## avg gene

### standard scaled

In [ ]:
fitness_df = avg_gene_standard_genn_fitness.copy()
rep_df = avg_gene_standard_genn_rep.copy()
alt_train_df = avg_gene_standard_genn_alt_train.copy()
alt_test_df = avg_gene_standard_genn_alt_test.copy()

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             important_90_n]
important[important['CV count'] >= 3]

In [ ]:
fitness_df = avg_gene_standard_gesr_fitness.copy()
rep_df = avg_gene_standard_gesr_rep.copy()
alt_train_df = avg_gene_standard_gesr_alt_train.copy()
alt_test_df = avg_gene_standard_gesr_alt_test.copy()

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             important_90_n]
important[important['CV count'] >= 3]

In [ ]:
avg_gene_standard_list = ['Avg Gene- Standard Scaled'] + genn_list + gesr_list

### minmax scaled

In [ ]:
fitness_df = avg_gene_minmax_genn_fitness.copy()
rep_df = avg_gene_minmax_genn_rep.copy()
alt_train_df = avg_gene_minmax_genn_alt_train.copy()
alt_test_df = avg_gene_minmax_genn_alt_test.copy()

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             important_90_n]
important[important['CV count'] >= 3]

In [ ]:
fitness_df = avg_gene_minmax_gesr_fitness.copy()
rep_df = avg_gene_minmax_gesr_rep.copy()
alt_train_df = avg_gene_minmax_gesr_alt_train.copy()
alt_test_df = avg_gene_minmax_gesr_alt_test.copy()

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             important_90_n]
important[important['CV count'] >= 3]

In [ ]:
avg_gene_minmax_list = ['Avg Gene- Minmax Scaled'] + genn_list + gesr_list

## positive control

### 100 features

#### 0-1 important

In [ ]:
fitness_df = pos_100_2_genn_fitness.copy()
rep_df = pos_100_2_genn_rep.copy()
alt_train_df = pos_100_2_genn_alt_train.copy()
alt_test_df = pos_100_2_genn_alt_test.copy()
features = {"feature_0", "feature_1"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
fitness_df = pos_100_2_gesr_fitness.copy()
rep_df = pos_100_2_gesr_rep.copy()
alt_train_df = pos_100_2_gesr_alt_train.copy()
alt_test_df = pos_100_2_gesr_alt_test.copy()
features = {"feature_0", "feature_1"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
pos_100_2_list = ['Positive Control- 100 Features, 0-1 with signal'] + genn_list + gesr_list

#### 0-2 important

In [ ]:
fitness_df = pos_100_3_genn_fitness.copy()
rep_df = pos_100_3_genn_rep.copy()
alt_train_df = pos_100_3_genn_alt_train.copy()
alt_test_df = pos_100_3_genn_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
fitness_df = pos_100_3_gesr_fitness.copy()
rep_df = pos_100_3_gesr_rep.copy()
alt_train_df = pos_100_3_gesr_alt_train.copy()
alt_test_df = pos_100_3_gesr_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
pos_100_3_list = ['Positive Control- 100 Features, 0-2 with signal'] + genn_list + gesr_list

#### 0-3 important

In [ ]:
fitness_df = pos_100_4_genn_fitness.copy()
rep_df = pos_100_4_genn_rep.copy()
alt_train_df = pos_100_4_genn_alt_train.copy()
alt_test_df = pos_100_4_genn_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2", "feature_3"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
fitness_df = pos_100_4_gesr_fitness.copy()
rep_df = pos_100_4_gesr_rep.copy()
alt_train_df = pos_100_4_gesr_alt_train.copy()
alt_test_df = pos_100_4_gesr_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2", "feature_3"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
pos_100_4_list = ['Positive Control- 100 Features, 0-3 with signal'] + genn_list + gesr_list

#### 0-4 important

In [ ]:
fitness_df = pos_100_5_genn_fitness.copy()
rep_df = pos_100_5_genn_rep.copy()
alt_train_df = pos_100_5_genn_alt_train.copy()
alt_test_df = pos_100_5_genn_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2", "feature_3", "feature_4"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
fitness_df = pos_100_5_gesr_fitness.copy()
rep_df = pos_100_5_gesr_rep.copy()
alt_train_df = pos_100_5_gesr_alt_train.copy()
alt_test_df = pos_100_5_gesr_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2", "feature_3", "feature_4"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
pos_100_5_list = ['Positive Control- 100 Features, 0-4 with signal'] + genn_list + gesr_list

### 2722 features

#### 0-1 important

In [ ]:
fitness_df = pos_2722_2_genn_fitness.copy()
rep_df = pos_2722_2_genn_rep.copy()
alt_train_df = pos_2722_2_genn_alt_train.copy()
alt_test_df = pos_2722_2_genn_alt_test.copy()
features = {"feature_0", "feature_1"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
fitness_df = pos_2722_2_gesr_fitness.copy()
rep_df = pos_2722_2_gesr_rep.copy()
alt_train_df = pos_2722_2_gesr_alt_train.copy()
alt_test_df = pos_2722_2_gesr_alt_test.copy()
features = {"feature_0", "feature_1"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
pos_2722_2_list = ['Positive Control- 18717 Features, 0-1 with signal'] + genn_list + gesr_list

#### 0-2 important

In [ ]:
fitness_df = pos_2722_3_genn_fitness.copy()
rep_df = pos_2722_3_genn_rep.copy()
alt_train_df = pos_2722_3_genn_alt_train.copy()
alt_test_df = pos_2722_3_genn_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
fitness_df = pos_2722_3_gesr_fitness.copy()
rep_df = pos_2722_3_gesr_rep.copy()
alt_train_df = pos_2722_3_gesr_alt_train.copy()
alt_test_df = pos_2722_3_gesr_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
pos_2722_3_list = ['Positive Control- 18717 Features, 0-2 with signal'] + genn_list + gesr_list

#### 0-3 important

In [ ]:
fitness_df = pos_2722_4_genn_fitness.copy()
rep_df = pos_2722_4_genn_rep.copy()
alt_train_df = pos_2722_4_genn_alt_train.copy()
alt_test_df = pos_2722_4_genn_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2", "feature_3"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
fitness_df = pos_2722_4_gesr_fitness.copy()
rep_df = pos_2722_4_gesr_rep.copy()
alt_train_df = pos_2722_4_gesr_alt_train.copy()
alt_test_df = pos_2722_4_gesr_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2", "feature_3"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
pos_2722_4_list = ['Positive Control- 18717 Features, 0-3 with signal'] + genn_list + gesr_list

#### 0-4 important

In [ ]:
fitness_df = pos_2722_5_genn_fitness.copy()
rep_df = pos_2722_5_genn_rep.copy()
alt_train_df = pos_2722_5_genn_alt_train.copy()
alt_test_df = pos_2722_5_genn_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2", "feature_3", "feature_4"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

genn_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
fitness_df = pos_2722_5_gesr_fitness.copy()
rep_df = pos_2722_5_gesr_rep.copy()
alt_train_df = pos_2722_5_gesr_alt_train.copy()
alt_test_df = pos_2722_5_gesr_alt_test.copy()
features = {"feature_0", "feature_1", "feature_2", "feature_3", "feature_4"}

train_ba = fitness_df['balanced_acc Training'].mean()
test_ba = fitness_df['Testing'].mean()

train_auroc = alt_train_df['auc'].mean()
test_auroc = alt_test_df['auc'].mean()

train_auprc = alt_train_df['auprc'].mean()
test_auprc = alt_test_df['auprc'].mean()

train_f1 = alt_train_df['f1_score'].mean()
test_f1 = alt_test_df['f1_score'].mean()

important = rep_df[['CV count', 'Variables']]
important['Variables'] = important['Variables'].str.split(' ')
important = important.explode('Variables')
important = important[important['Variables'].str.strip() != '']

important_dict = important['CV count'].value_counts(dropna = False).to_dict()
for cv in list(range(1, 6)):
    if important.loc[important['CV count'] == cv, 'Variables'].isna().all():
        important_dict[cv] = np.nan
    else:
        continue
important_dict = dict(sorted(important_dict.items()))

important_90_n = np.nan_to_num(important_dict[4]) + np.nan_to_num(important_dict[5])

all_bool = features.issubset(important["Variables"])

important_90 = important[important['CV count'] >= 4]
important_bool = features.issubset(important_90["Variables"])

nonexpected_important_bool = set(important_90['Variables'].dropna()) - set(features)

expected = important[important['Variables'].isin(features)]
expected_dict = dict(zip(expected['Variables'], expected['CV count']))
expected_dict = dict(sorted(expected_dict.items()))

gesr_list = [train_ba,
             test_ba,
             train_auroc,
             test_auroc,
             train_auprc,
             test_auprc,
             train_f1,
             test_f1,
             important_dict,
             expected_dict,
             important_90_n,
             all_bool,
             important_bool,
             nonexpected_important_bool]

In [ ]:
pos_2722_5_list = ['Positive Control- 18717 Features, 0-4 with signal'] + genn_list + gesr_list

# create summary dataframe

## biological data

In [ ]:
bio_summary_df = pd.DataFrame([avg_gene_standard_list,
                               avg_pathway_standard_list,
                               avg_gene_minmax_list,
                               avg_pathway_minmax_list],
                               columns = ['Biological Data',
                                          'genn training BA',
                                          'genn testing BA',
                                          'genn training AUROC',
                                          'genn testing AUROC',
                                          'genn training AUPRC',
                                          'genn testing AUPRC',
                                          'genn training F1',
                                          'genn testing F1',
                                          'genn N feature replications',
                                          'genn N features in 80% CVs',
                                          'gesr training BA',
                                          'gesr testing BA',
                                          'gesr training AUROC',
                                          'gesr testing AUROC',
                                          'gesr training AUPRC',
                                          'gesr testing AUPRC',
                                          'gesr training F1',
                                          'gesr testing F1',
                                          'gesr N feature replications',
                                          'gesr N features in 80% CVs'])
bio_summary_df

## simulated data

In [ ]:
summary_df = pd.DataFrame([neg_2722_list,
                           pos_2722_2_list,
                           pos_2722_3_list,
                           pos_2722_4_list,
                           pos_2722_5_list],
                           columns = ['Simulated Data',
                                      'genn training BA',
                                      'genn testing BA',
                                      'genn training AUROC',
                                      'genn testing AUROC',
                                      'genn training AUPRC',
                                      'genn testing AUPRC',
                                      'genn training F1',
                                      'genn testing F1',
                                      'genn N feature replications',
                                      'genn N expected feature replications',
                                      'genn N features in 80% CVs',
                                      'genn expected important features identified in any amount of CVs',
                                      'genn expected important features identified in 80% CVs',
                                      'genn non-expected important features identified in 80% CVs',
                                      'gesr training BA',
                                      'gesr testing BA',
                                      'gesr training AUROC',
                                      'gesr testing AUROC',
                                      'gesr training AUPRC',
                                      'gesr testing AUPRC',
                                      'gesr training F1',
                                      'gesr testing F1',
                                      'gesr N feature replications',
                                      'gesr N expected feature replications',
                                      'gesr N features in 80% CVs',
                                      'gesr expected important features identified in any amount of CVs',
                                      'gesr expected important features identified in 80% CVs',
                                      'gesr non-expected important features identified in 80% CVs'])
summary_df

# add in extra data

## run time

### biological data

In [ ]:
jobid = '44946589'
num_gens = 600

genn_runtime = []
gesr_runtime = []
genn_max_fitness = []
gesr_max_fitness = []
genn_fitness_drop_list = []
gesr_fitness_drop_list = []
genn_fitness_list = []
gesr_fitness_list = []

for id in list(range(1, 9)):
    filename = 'ML/athena/logs/athena.' + jobid + '-' + str(id) + '.out'
    runtime_found = False
    genn_fitness_found = False
    gesr_fitness_found = False
    genn_fitness = {}
    gesr_fitness = {}
    with open(filename, "r") as f:
        for line in f:
            if "Run time" in line:
                line = int(re.search(r"[0-9]+", line).group())
                if id <= 4:
                    genn_runtime.append(line)
                elif id >= 5:
                    gesr_runtime.append(line)
                runtime_found = True
                continue
            if 'CV:' in line:
                line = int(re.search(r"[0-9]+", line).group())
                cv = line
                if id <= 4:
                    genn_fitness[cv] = {}
                elif id >= 5:
                    gesr_fitness[cv] = {}
                continue
            if 'gen =' in line:
                gen = int(re.search(r'gen\s*=\s*(\d+)', line).group(1))
                fitness = float(re.search(r'Best fitness\s*=\s*([\d\.]+)', line).group(1))
                if id <= 4:
                    genn_fitness[cv][gen] = fitness
                    if len(genn_fitness) == 5:
                        if sum(len(subdict) for subdict in genn_fitness.values()) == num_gens:
                            genn_fitness_found = True
                elif id >= 5:
                    gesr_fitness[cv][gen] = fitness
                    if len(gesr_fitness) == 5:
                        if sum(len(subdict) for subdict in gesr_fitness.values()) == num_gens:
                            gesr_fitness_found = True
            if runtime_found and genn_fitness_found and gesr_fitness_found:
                continue

    fitness_drop_dict = {}
    if genn_fitness:
        genn_fitness_list.append(genn_fitness)
    if gesr_fitness:
        gesr_fitness_list.append(gesr_fitness)
    if id <= 4:
        max_per_cv = {cv: max(gens.items(), key = lambda x: x[1]) for cv, gens in genn_fitness.items()}
        max_per_cv = {cv: (gen, round(fitness, 3)) for cv, (gen, fitness) in max_per_cv.items()}
        genn_max_fitness.append(max_per_cv)
        for cv, (max_gen, max_fitness) in max_per_cv.items():
            gens = genn_fitness[cv]
            later_fitness = [gens[g] for g in sorted(gens) if g > max_gen]
            fitness_drop_dict[cv] = any(f < max_fitness for f in later_fitness)
        genn_fitness_drop_list.append(fitness_drop_dict)
    elif id >= 5:
        max_per_cv = {cv: max(gens.items(), key = lambda x: x[1]) for cv, gens in gesr_fitness.items()}
        max_per_cv = {cv: (gen, round(fitness, 3)) for cv, (gen, fitness) in max_per_cv.items()}
        gesr_max_fitness.append(max_per_cv)
        for cv, (max_gen, max_fitness) in max_per_cv.items():
            gens = gesr_fitness[cv]
            later_fitness = [gens[g] for g in sorted(gens) if g > max_gen]
            fitness_drop_dict[cv] = any(f < max_fitness for f in later_fitness)
        gesr_fitness_drop_list.append(fitness_drop_dict)    

In [ ]:
bio_summary_df.insert(11, 'genn max fitness generation per CV', genn_max_fitness)
bio_summary_df.insert(12, 'genn fitness decrease after max per CV', genn_fitness_drop_list)
bio_summary_df.insert(13, 'genn Run Time', genn_runtime)
bio_summary_df['gesr max fitness generation per CV'] = gesr_max_fitness
bio_summary_df['gesr fitness decrease after max per CV'] = gesr_fitness_drop_list
bio_summary_df['gesr Run Time'] = gesr_runtime
bio_summary_df.head()

In [ ]:
bio_summary_df['genn Run Time'] = pd.to_timedelta(bio_summary_df['genn Run Time'], unit = 's')
bio_summary_df['gesr Run Time'] = pd.to_timedelta(bio_summary_df['gesr Run Time'], unit = 's')
bio_summary_df = bio_summary_df.round(3)
bio_summary_df.head()

### simulated data

In [ ]:
jobid = '45790960'
num_gens = 250

genn_runtime = []
gesr_runtime = []
genn_max_fitness = []
gesr_max_fitness = []
genn_fitness_drop_list = []
gesr_fitness_drop_list = []
genn_fitness_list = []
gesr_fitness_list = []

for id in list(range(1, 81)):
    filename = 'ML/athena/logs/athena_simulations.' + jobid + '-' + str(id) + '.out'
    runtime_found = False
    fitness_found = False
    genn_fitness = {}
    gesr_fitness = {}
    with open(filename, "r") as f:  
        content = f.read()
        if "Exit" in content:
            print(f"Skipping failed job: {filename}")
            if id in list(range(1, 9)) + list(range(17, 25)) + list(range(33, 41)) + list(range(49, 57)) + list(range(65, 73)):
                genn_max_fitness.append(np.nan)
                genn_fitness_drop_list.append(np.nan)
                genn_runtime.append(np.nan)
            else:
                gesr_max_fitness.append(np.nan)
                gesr_fitness_drop_list.append(np.nan)
                gesr_runtime.append(np.nan)
            continue
        f.seek(0)
        for line in f:
            if "Run time" in line:
                line = int(re.search(r"[0-9]+", line).group())
                if id in list(range(1, 9)) + list(range(17, 25)) + list(range(33, 41)) + list(range(49, 57)) + list(range(65, 73)):
                    genn_runtime.append(line)
                else:
                    gesr_runtime.append(line)
                runtime_found = True
                continue
            if 'CV:' in line:
                line = int(re.search(r"[0-9]+", line).group())
                cv = line
                if id in list(range(1, 9)) + list(range(17, 25)) + list(range(33, 41)) + list(range(49, 57)) + list(range(65, 73)):
                    genn_fitness[cv] = {}
                else:
                    gesr_fitness[cv] = {}
                continue
            if 'gen =' in line:
                gen = int(re.search(r'gen\s*=\s*(\d+)', line).group(1))
                fitness = float(re.search(r'Best fitness\s*=\s*([\d\.]+)', line).group(1))
                if id in list(range(1, 9)) + list(range(17, 25)) + list(range(33, 41)) + list(range(49, 57)) + list(range(65, 73)):
                    genn_fitness[cv][gen] = fitness
                    if len(genn_fitness) == 5:
                        if sum(len(subdict) for subdict in genn_fitness.values()) == num_gens:
                            fitness_found = True
                else:
                    gesr_fitness[cv][gen] = fitness
                    if len(gesr_fitness) == 5:
                        if sum(len(subdict) for subdict in gesr_fitness.values()) == num_gens:
                            fitness_found = True
            if runtime_found and fitness_found:
                continue

    fitness_drop_dict = {}
    genn_fitness_list.append(genn_fitness)
    gesr_fitness_list.append(gesr_fitness)
    if id in list(range(1, 9)) + list(range(17, 25)) + list(range(33, 41)) + list(range(49, 57)) + list(range(65, 73)):
        max_per_cv = {cv: max(gens.items(), key = lambda x: x[1]) for cv, gens in genn_fitness.items()}
        max_per_cv = {cv: (gen, round(fitness, 3)) for cv, (gen, fitness) in max_per_cv.items()}
        genn_max_fitness.append(max_per_cv)
        for cv, (max_gen, max_fitness) in max_per_cv.items():
            gens = genn_fitness[cv]
            later_fitness = [gens[g] for g in sorted(gens) if g > max_gen]
            fitness_drop_dict[cv] = any(f < max_fitness for f in later_fitness)
        genn_fitness_drop_list.append(fitness_drop_dict)
    else:
        max_per_cv = {cv: max(gens.items(), key = lambda x: x[1]) for cv, gens in gesr_fitness.items()}
        max_per_cv = {cv: (gen, round(fitness, 3)) for cv, (gen, fitness) in max_per_cv.items()}
        gesr_max_fitness.append(max_per_cv)
        for cv, (max_gen, max_fitness) in max_per_cv.items():
            gens = gesr_fitness[cv]
            later_fitness = [gens[g] for g in sorted(gens) if g > max_gen]
            fitness_drop_dict[cv] = any(f < max_fitness for f in later_fitness)
        gesr_fitness_drop_list.append(fitness_drop_dict)    

In [ ]:
len(genn_fitness_list)

In [ ]:
len(gesr_fitness_list)

In [ ]:
sim_wide['genn max fitness generation per CV'] = genn_max_fitness
sim_wide['gesr max fitness generation per CV'] = gesr_max_fitness
sim_wide['genn fitness decrease after max per CV'] = genn_fitness_drop_list
sim_wide['gesr fitness decrease after max per CV'] = gesr_fitness_drop_list
sim_wide['genn Run Time'] = genn_runtime
sim_wide['gesr Run Time'] = gesr_runtime
sim_wide.head()

In [ ]:
sim_wide['genn Run Time'] = pd.to_timedelta(sim_wide['genn Run Time'], unit = 's')
sim_wide['gesr Run Time'] = pd.to_timedelta(sim_wide['gesr Run Time'], unit = 's')
sim_wide = sim_wide.round(3)
sim_wide.head()

## eval df

In [ ]:
sim_wide['Simulated Data'] = sim_wide['Simulated Data'].str.replace('negative_control', 'Negative Control')
sim_wide['Simulated Data'] = sim_wide['Simulated Data'].str.replace('features_0-1_signal.positive_control', 'Positive Control- 0-1 with signal')
sim_wide['Simulated Data'] = sim_wide['Simulated Data'].str.replace('features_0-2_signal.positive_control', 'Positive Control- 0-2 with signal')
sim_wide['Simulated Data'] = sim_wide['Simulated Data'].str.replace('features_0-3_signal.positive_control', 'Positive Control- 0-3 with signal')
sim_wide['Simulated Data'] = sim_wide['Simulated Data'].str.replace('features_0-4_signal.positive_control', 'Positive Control- 0-4 with signal')

In [ ]:
eval_df = eval_df.round(3)

In [ ]:
summary_df = eval_df.merge(sim_wide, on = ['Simulated Data', 'pval'])
print(summary_df.shape)
summary_df.head()

# split dfs for ppt

## biological

In [ ]:
bio_summary_df.columns

In [ ]:
bio_summary_df_1 = bio_summary_df[['Biological Data', 'genn training BA', 'genn testing BA','genn Run Time','gesr training BA', 'gesr testing BA','gesr Run Time']]
bio_summary_df_1

In [ ]:
bio_summary_df_2 = bio_summary_df[['Biological Data',
                                   'genn max fitness generation per CV',
                                   'genn fitness decrease after max per CV',
                                   'genn Run Time',
                                   'genn N features in 80% CVs',
                                   'gesr N features in 80% CVs',
                                   'gesr max fitness generation per CV',
                                   'gesr fitness decrease after max per CV']]
bio_summary_df_2

## simulated

In [ ]:
summary_df.columns

In [ ]:
summary_df_1 = summary_df[['Simulated Data',
                           'pval',
                           'LR BA',
                           'training BA_genn',
                           'testing BA_genn',
                           'training BA_gesr',
                           'testing BA_gesr',
                           'LR AUROC',
                           'training AUROC_genn',
                           'testing AUROC_genn',
                           'training AUROC_gesr',
                           'testing AUROC_gesr']]
summary_df_1

In [ ]:
summary_df_2 = summary_df[['Simulated Data',
                           'pval',
                           'LR AUPRC',
                           'training AUPRC_genn',
                           'testing AUPRC_genn',
                           'training AUPRC_gesr',
                           'testing AUPRC_gesr',
                           'LR F1',
                           'training F1_genn',
                           'testing F1_genn',
                           'training F1_gesr',
                           'testing F1_gesr']]
summary_df_2

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
summary_df_3 = summary_df[['Simulated Data', 
                           'pval',
                           'N feature replications_genn',
                           'N expected feature replications_genn',
                           'N features in 80% CVs_genn',
                           'expected important features identified in any amount of CVs_genn',
                           'expected important features identified in 80% CVs_genn',
                           'non-expected important features identified in 80% CVs_genn',
                           'N feature replications_gesr',
                           'N expected feature replications_gesr',
                           'N features in 80% CVs_gesr',
                           'expected important features identified in any amount of CVs_gesr',
                           'expected important features identified in 80% CVs_gesr',
                           'non-expected important features identified in 80% CVs_gesr']]
summary_df_3

In [ ]:
summary_df_4 = summary_df[['Simulated Data',
                           'pval',
                           'genn max fitness generation per CV',
                           'genn fitness decrease after max per CV',
                           'genn Run Time',
                           'gesr max fitness generation per CV',
                           'gesr fitness decrease after max per CV',
                           'gesr Run Time']]
summary_df_4

# make gens by BA graph

## biological data

In [ ]:
titles = bio_summary_df[['Biological Data']]

for i in list(range(0, 4)):

    #gesr_num = i + 4
    
    genn_nested_dict = genn_fitness_list[i]
    gesr_nested_dict = gesr_fitness_list[i]
    
    genn_fitness_df = pd.DataFrame.from_dict(genn_nested_dict, orient = 'index').transpose()
    gesr_fitness_df = pd.DataFrame.from_dict(gesr_nested_dict, orient = 'index').transpose()
    
    genn_fitness_df['GENERATIONS'] = genn_fitness_df.index
    gesr_fitness_df['GENERATIONS'] = gesr_fitness_df.index

    genn_plot_input = genn_fitness_df.reset_index(drop = True).melt(id_vars = 'GENERATIONS', var_name = 'CV', value_name = 'BA')
    gesr_plot_input = gesr_fitness_df.reset_index(drop = True).melt(id_vars = 'GENERATIONS', var_name = 'CV', value_name = 'BA')

    genn_plot_input['CV'] = 'CV = ' + genn_plot_input['CV'].astype(str) + '; grammar = genn'
    gesr_plot_input['CV'] = 'CV = ' + gesr_plot_input['CV'].astype(str) + '; grammar = gesr'

    plot_input = pd.concat([genn_plot_input, gesr_plot_input], axis = 0)

    title = titles.loc[i, 'Biological Data']
    title_no_space = title.replace(' ', '_')
    title_no_space = title_no_space.replace('-', '')

    plt.figure(figsize = (16, 8))
    sns.lineplot(data = plot_input, x = 'GENERATIONS', y = 'BA', hue = 'CV', marker = 'o')
    plt.xticks(rotation = 45, ha = 'right')
    plt.title(title)
    plt.legend(title = 'CV and Grammar', bbox_to_anchor = (1.05, 1), loc = 'upper left')
    plt.ylabel('BA')
    plt.axvline(x = 400, color = 'gray', linestyle = '--', alpha = 0.5)
    plt.xticks(range(0, 601, 25))
    plt.tight_layout()
    plt.savefig(('ML/athena/plots/ADSP.biological.athena.' + title_no_space + '.600_gens.100000_popsize.crossover_block_100_gens.min_init_tree_depth_11.max_init_tree_depth_15.max_depth_50.line_graph.png'),
                dpi = 300, bbox_inches = 'tight')

## simulated data

In [ ]:
from itertools import product

In [ ]:
print(len(genn_fitness_list))

In [ ]:
print(len(gesr_fitness_list))

In [ ]:
titles = [f"{a} {b}" for a, b in zip(sim_wide['Simulated Data'], sim_wide['pval'])]
genn_ids = list(range(1, 9)) + list(range(17, 25)) + list(range(33, 41)) + list(range(49, 57)) + list(range(65, 73))
gesr_ids = [i for i in range(1, 81) if i not in genn_ids]

genn_map = {file_id: idx for idx, file_id in enumerate(genn_ids)}
gesr_map = {file_id: idx for idx, file_id in enumerate(gesr_ids)}

for i in list(range(16, 32)):

    gesr_num = i + 8
    
    gesr_file_id = i + 8  # adjust if needed
    gesr_index = gesr_map[gesr_file_id]

    genn_nested_dict = genn_fitness_list[genn_index]
    gesr_nested_dict = gesr_fitness_list[gesr_index]

    print(len(genn_fitness_list[i]))
    print(len(gesr_fitness_list[gesr_index]))
    
    genn_fitness_df = pd.DataFrame.from_dict(genn_nested_dict, orient = 'index').transpose()
    gesr_fitness_df = pd.DataFrame.from_dict(gesr_nested_dict, orient = 'index').transpose()
    
    genn_fitness_df['GENERATIONS'] = genn_fitness_df.index
    gesr_fitness_df['GENERATIONS'] = gesr_fitness_df.index

    genn_plot_input = genn_fitness_df.reset_index(drop = True).melt(id_vars = 'GENERATIONS', var_name = 'CV', value_name = 'BA')
    gesr_plot_input = gesr_fitness_df.reset_index(drop = True).melt(id_vars = 'GENERATIONS', var_name = 'CV', value_name = 'BA')

    genn_plot_input['CV'] = 'CV = ' + genn_plot_input['CV'].astype(str) + '; grammar = genn'
    gesr_plot_input['CV'] = 'CV = ' + gesr_plot_input['CV'].astype(str) + '; grammar = gesr'

    plot_input = pd.concat([genn_plot_input, gesr_plot_input], axis = 0)
    print(len(plot_input.index))

    #title = titles[i]
    #title_no_space = title.replace(' ', '_')
    #title_no_space = title_no_space.replace('-', '')

    #plt.figure(figsize = (16, 8))
    #sns.lineplot(data = plot_input, x = 'GENERATIONS', y = 'BA', hue = 'CV', marker = 'o')
    #plt.xticks(rotation = 45, ha = 'right')
    #plt.title(title)
    #plt.legend(title = 'CV and Grammar', bbox_to_anchor = (1.05, 1), loc = 'upper left')
    #plt.ylabel('BA')
    #plt.axvline(x = 400, color = 'gray', linestyle = '--', alpha = 0.5)
    #plt.xticks(range(0, 251, 25))
    #plt.tight_layout()
    #plt.savefig(('ML/athena/plots/ADSP.simulated.athena.' + title_no_space + '.250_gens.30000_popsize.max_depth_50.line_graph.png'),
    #            dpi = 300, bbox_inches = 'tight')

In [ ]:
titles = [f"{a} {b}" for a, b in zip(sim_wide['Simulated Data'], sim_wide['pval'])]

for i in list(range(0, 40)):

    gesr_num = i + 8
    
    genn_nested_dict = genn_fitness_list[i]
    gesr_nested_dict = gesr_fitness_list[gesr_num]
    
    genn_fitness_df = pd.DataFrame.from_dict(genn_nested_dict, orient = 'index').transpose()
    gesr_fitness_df = pd.DataFrame.from_dict(gesr_nested_dict, orient = 'index').transpose()
    
    genn_fitness_df['GENERATIONS'] = genn_fitness_df.index
    gesr_fitness_df['GENERATIONS'] = gesr_fitness_df.index

    genn_plot_input = genn_fitness_df.reset_index(drop = True).melt(id_vars = 'GENERATIONS', var_name = 'CV', value_name = 'BA')
    gesr_plot_input = gesr_fitness_df.reset_index(drop = True).melt(id_vars = 'GENERATIONS', var_name = 'CV', value_name = 'BA')

    genn_plot_input['CV'] = 'CV = ' + genn_plot_input['CV'].astype(str) + '; grammar = genn'
    gesr_plot_input['CV'] = 'CV = ' + gesr_plot_input['CV'].astype(str) + '; grammar = gesr'

    plot_input = pd.concat([genn_plot_input, gesr_plot_input], axis = 0)
    print(len(plot_input.index))

    title = titles[i]
    title_no_space = title.replace(' ', '_')
    title_no_space = title_no_space.replace('-', '')

    plt.figure(figsize = (16, 8))
    sns.lineplot(data = plot_input, x = 'GENERATIONS', y = 'BA', hue = 'CV', marker = 'o')
    plt.xticks(rotation = 45, ha = 'right')
    plt.title(title)
    plt.legend(title = 'CV and Grammar', bbox_to_anchor = (1.05, 1), loc = 'upper left')
    plt.ylabel('BA')
    #plt.axvline(x = 400, color = 'gray', linestyle = '--', alpha = 0.5)
    plt.xticks(range(0, 251, 25))
    plt.tight_layout()
    plt.savefig(('ML/athena/plots/ADSP.simulated.athena.' + title_no_space + '.250_gens.30000_popsize.max_depth_50.line_graph.png'),
                dpi = 300, bbox_inches = 'tight')

# export

## simulated

In [ ]:
summary_df.to_csv('ML/athena/summary/ADSP.simulated.18717_features.250_gens.30000_popsize.max_depth_50.summary.csv',
                  index = None,
                  na_rep = 'NaN')

In [ ]:
summary_df_1.to_csv('ML/athena/summary/ADSP.simulated.18717_features.250_gens.30000_popsize.max_depth_50.BA_AUROC.summary.csv',
                  index = None,
                  na_rep = 'NaN')

In [ ]:
summary_df_2.to_csv('ML/athena/summary/ADSP.simulated.18717_features.250_gens.30000_popsize.max_depth_50.AUPRC_F1.summary.csv',
                  index = None,
                  na_rep = 'NaN')

In [ ]:
summary_df_3.to_csv('ML/athena/summary/ADSP.simulated.18717_features.250_gens.30000_popsize.max_depth_50.features.summary.csv',
                  index = None,
                  na_rep = 'NaN')

In [ ]:
summary_df_4.to_csv('ML/athena/summary/ADSP.simulated.18717_features.250_gens.30000_popsize.max_depth_50.gen_runtime.summary.csv',
                  index = None,
                  na_rep = 'NaN')

## biological

In [ ]:
bio_summary_df.to_csv('ML/athena/summary/ADSP.biological.600_gens.100000_popsize.crossover_block_200_gens.min_init_tree_depth_11.max_init_tree_depth_15.max_depth_50.summary.csv',
                      index = None,
                      na_rep = 'NaN')

In [ ]:
bio_summary_df_1.to_csv('ML/athena/summary/ADSP.biological.600_gens.100000_popsize.crossover_block_200_gens.min_init_tree_depth_11.max_init_tree_depth_15.max_depth_50.BA_runtime.summary.csv',
                      index = None,
                      na_rep = 'NaN')

In [ ]:
bio_summary_df_2.to_csv('ML/athena/summary/ADSP.biological.600_gens.100000_popsize.max_depth_50.crossover_block_50_gens.min_init_tree_depth_11.max_init_tree_depth_15.max_depth_50.features.summary.csv',
                      index = None,
                      na_rep = 'NaN')